# Fine-tuning the Qeema matcher on Colab's free tier

`multilingual-e5-base` is a general web-text model being asked about Libyan
dialect and franco-arabe — `9arora gaz`, `دحي وطني`, `بمبلة`, `بانقة`. This
notebook measures whether fine-tuning it on the project's own vocabulary helps,
and by how much.

**Run it on a GPU.** Runtime → Change runtime type → T4 GPU. The free tier is
enough: the dataset is small and a run takes a few minutes. On a laptop CPU the
same run takes hours and makes the machine unusable.

Nothing here is proprietary. The base model is MIT, the data is in the public
repository, and the output is a set of weights you can host yourself — which is
the point, because constraint C1 forbids a paid or hosted service anywhere in
the runtime path.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU — set Runtime > Change runtime type > T4 GPU, or this will take hours"

In [ ]:
!pip install -q "sentence-transformers>=3" scikit-learn pyyaml
!git clone --depth 1 https://github.com/Binary-ly/qeema.git
%cd qeema/ml

## The experiment

`embedding_finetune.py` splits every item's wordings 60/40, deterministically.
The index holds the catalogue plus the training 60%; the queries are the held-out
40%, which appear nowhere in the index. It reports two numbers, not one:

- **top-1 retrieval** — did the nearest indexed passage belong to the right item
- **distractor separation (AUC)** — the gap between how close a held-out correct
  wording sits to its item and how close a distractor sits to whatever it is
  nearest

The second one matters because retrieval accuracy is easy to buy by making
everything look like everything else, which would destroy the platform's ability
to refuse a product that is not in the basket. A model that improves top-1 while
closing that gap has not improved anything.

In [ ]:
import os
os.environ["QEEMA_FINETUNE_OUT"] = "/content/e5-qeema-final"
# os.environ["FULL_SWEEP"] = "1"   # four configurations instead of one; GPU only
!python scripts/embedding_finetune.py

## Did it help on real shop text?

The run above scores against the project's own corpus. This scores the fine-tuned
model against `ml/data/real-text/` — wordings collected from a government price
bulletin, WFP, and Libyan shop pages, none of which anyone here wrote.

In [ ]:
import json, pathlib
cfg = pathlib.Path("src/qeema_ml/config.py")
print("point the service at the fine-tuned weights by setting:")
print('  QEEMA_ML_EMBEDDING_MODEL=/content/e5-qeema-final   (or a Hugging Face repo id)')

# baseline, then fine-tuned, on the same held-out real text
!QEEMA_ML_EMBEDDING_MODEL=intfloat/multilingual-e5-base python scripts/real_text_evaluation.py
!QEEMA_ML_EMBEDDING_MODEL=/content/e5-qeema-final python scripts/real_text_evaluation.py

## Keeping the weights

Only publish if the numbers above justify it — a model that trades distractor
separation for top-1 is worse for this platform even when the headline improves.

```python
from huggingface_hub import notebook_login; notebook_login()
from sentence_transformers import SentenceTransformer
SentenceTransformer("/content/e5-qeema-final").push_to_hub("<your-org>/e5-qeema-ly")
```

Then set `QEEMA_ML_EMBEDDING_MODEL` to that repo id. It stays self-hostable and
openly licensed, so `docker compose up` still works for someone with no
accounts anywhere.